# Khung Thực Nghiệm Đánh Giá Đặc Tính Thuật Toán ApEn

#### Giai đoạn 1: Khởi tạo các mô hình dữ liệu chuẩn (Benchmark Datasets)
Mục tiêu là xây dựng các hàm sinh dữ liệu với các mức độ động lực học khác nhau để kiểm chứng nguyên lý: **Tất định < Hỗn loạn < Ngẫu nhiên**.

*   **Hệ Tất định (Pure Sine):** Chuỗi tuần hoàn hoàn hảo.
    $$X_j = \sqrt{2} \sin\left(\frac{2\pi j}{12}\right)$$
    *(Hệ số $\sqrt{2}$ đảm bảo phương sai của chuỗi bằng $1$)*
*   **Hệ Hỗn loạn (Logistic Map):** Trạng thái hỗn loạn hoàn toàn (fully chaotic).
    $$L_{j+1} = 4 L_j (1 - L_j), \quad L_0 \in (0, 1)$$
*   **Hệ Ngẫu nhiên (White Noise):** Chuỗi i.i.d phân bố đều.
    $$W_j \sim \mathcal{U}[-\sqrt{3}, \sqrt{3}]$$
    *(Khoảng $[-\sqrt{3}, \sqrt{3}]$ đảm bảo kỳ vọng $\mu = 0$ và phương sai $\sigma^2 = 1$)*
*   **Hệ Pha trộn MIX(p) (Theo Pincus 1991):** 
    Sử dụng phân bố Bernoulli để điều khiển xác suất $p$ chuyển đổi giữa tín hiệu tất định ($X_j$) và ngẫu nhiên ($Y_j$).
    $$Z_j \sim \text{Bernoulli}(p) \implies P(Z_j = 1) = p, \quad P(Z_j = 0) = 1 - p$$
    $$MIX_j = (1 - Z_j)X_j + Z_j W_j$$
    *Sanity check:* Đảm bảo đầu ra của mọi kịch bản đều tuân thủ $ApEn(X) < ApEn(L) < ApEn(W)$.

#### Giai đoạn 2: Tái tạo Hình 1 bằng Mô phỏng Monte Carlo
Thiết lập lưới tham số để kiểm tra giới hạn hội tụ của dữ liệu ngắn thông qua xu hướng thống kê, loại bỏ nhiễu do random seed.

*   **Không gian tham số (Parameter Grid):**
    *   Lưới nhiễu: $p \in \{0, 0.01, 0.02, \dots, 1.0\}$ (101 điểm).
    *   Độ dài chuỗi: $N \in \{300, 1000, 3000\}$.
    *   Hệ số ApEn cố định: $m = 2, r = 0.18$.
*   **Vòng lặp Monte Carlo:** 
    Với mỗi cấu hình $(p, N)$, sinh độc lập $K = 30$ chuỗi $MIX^{(k)}$ và tính $ApEn_k$ tương ứng.
*   **Toán học hóa quá trình tổng hợp thống kê:**
    *   Kỳ vọng mẫu (Mean):
        $$\mu_{ApEn}(p) = \frac{1}{K} \sum_{k=1}^{K} ApEn_k(p)$$
    *   Độ lệch chuẩn mẫu (Standard Deviation) để đo lường dao động:
        $$\sigma_{ApEn}(p) = \sqrt{\frac{1}{K-1} \sum_{k=1}^{K} \left( ApEn_k(p) - \mu_{ApEn}(p) \right)^2}$$
*   **Kết xuất đồ thị:** Vẽ dải băng sai số $\mu_{ApEn}(p) \pm \sigma_{ApEn}(p)$ theo trục hoành $p$. 

#### Giai đoạn 3: Phân tích Độ nhạy (Sensitivity Analysis)
Định lượng mức độ phụ thuộc của thuật toán vào các tham số đầu vào và khả năng chịu đựng nhiễu nền.

*   **Thí nghiệm 3A: Khảo sát dung sai $r$ (Trade-off Analysis)**
    *   Cố định $N = 1000, m = 2$.
    *   Quét $r \in \{0.05, 0.10, 0.15, 0.20, 0.25, 0.30\}$.
    *   *Kỳ vọng:* Vẽ sự suy giảm của $\mu_{ApEn}$ và sự phình to của $\sigma_{ApEn}$ khi $r \to 0$.
*   **Thí nghiệm 3B: Khảo sát số chiều nhúng $m$**
    *   Cố định $N = 1000, r = 0.20$.
    *   Quét $m \in \{1, 2, 3, 4\}$.
    *   *Kỳ vọng:* Chứng minh tốc độ triệt tiêu số lượng vector láng giềng khi không gian pha mở rộng, củng cố cơ sở chọn $m=2$.
*   **Thí nghiệm 3C: Khả năng chống nhiễu cộng tính (Additive Noise Robustness)**
    *   Mô phỏng nhiễu đo lường (Measurement Noise) tác động lên nền tín hiệu tuần hoàn:
        $$\eta_j \sim \mathcal{N}(0, \sigma_{\eta}^2)$$
        $$S_j = \sqrt{2} \sin\left(\frac{2\pi j}{12}\right) + \eta_j$$
    *   Quét mức độ nhiễu $\sigma_{\eta} \in \{0, 0.01, 0.05, 0.10, 0.20\}$.
    *   *Kỳ vọng:* Định lượng "ngưỡng vỡ" của thuật toán — mức $\sigma_{\eta}$ mà tại đó ApEn bắt đầu mất tính ổn định và tăng vọt, báo hiệu thuật toán bắt đầu khớp vào nhiễu thay vì cấu trúc hệ thống.